# b1AR system simplification

Fitting sensitivity to ISO.

In [ ]:
using ModelingToolkit
using OrdinaryDiffEq, SteadyStateDiffEq, DiffEqCallbacks
using Plots
using CurveFit
using Model
using Model: μM, hil, Hz, hilr, second
Plots.default(lw=1.5)

## Setup b1AR system

In [ ]:
@parameters ATP = 5000μM ISO = 0μM
@time "Build system" @mtkcompile sys = Model.get_bar_sys(ATP, ISO)
@time "Build problem" prob = SteadyStateProblem(sys, [])

alg = DynamicSS(KenCarp47())
@time "Solve problem" sol = solve(prob, alg; abstol=1e-8, reltol=1e-8) ## warmup

In [ ]:
@unpack cAMP, fracPKACI, fracPKACII, fracPP1, fracPLBp, fracPLMp, TnI_PKAp, LCCa_PKAp, LCCb_PKAp, IKUR_PKAp, RyR_PKAp = sys

tracked_states = [cAMP, fracPKACI, fracPKACII, fracPP1, fracPLBp, fracPLMp, TnI_PKAp, LCCa_PKAp, LCCb_PKAp, IKUR_PKAp, RyR_PKAp]
stimap = Dict(k=>i for (i, k) in enumerate(tracked_states))

sol[tracked_states]

## Ensemble simulations

Log scale for ISO concentration.

In [ ]:
iso = exp10.(range(log10(1e-4μM), log10(1μM), length=1001))
trajectories=length(iso)
prob_func = (prob, i, repeat) -> remake(prob, p=[ISO => iso[i]])
output_func = (sol, i) -> (sol[tracked_states], false)
eprob = EnsembleProblem(prob; prob_func, output_func)

In [ ]:
@time "Solve problem" sim = solve(eprob, alg; trajectories, abstol=1e-8, reltol=1e-8);